# Review references

This notebook will audit validation and test commit subjects against their diffs and explore the prepared data.

## Import dependencies

In [ ]:
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq
from whats_that_code.election import guess_language_all_methods

## Configure review paths

Set the directory containing the prepared validation and test splits and the private sidecar location for review decisions.

In [ ]:
prepared_directory = Path("prepared")
review_path = prepared_directory / "review.parquet"

## Load the split datasets

This block loads the prepared validation and test Parquet files.

In [ ]:
validation_path = prepared_directory / "validation.parquet"
test_path = prepared_directory / "test.parquet"
if not validation_path.is_file() or not test_path.is_file():
    raise FileNotFoundError("Prepared validation and test Parquet files are required.")

validation_table = pq.read_table(validation_path)
test_table = pq.read_table(test_path)

## Validate the reference columns

The review needs an identifier, subject, diff, commit date, and repository group. This block checks that both held-out splits contain those columns and converts them into rows.

In [ ]:
reference_columns = (
    "id",
    "repository_group_id",
    "subject",
    "diff",
    "committed_at_utc",
)
for split_name, table in (("validation", validation_table), ("test", test_table)):
    missing_columns = [column for column in reference_columns if column not in table.column_names]
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {', '.join(missing_columns)}")

validation_rows = validation_table.select(list(reference_columns)).to_pylist()
test_rows = test_table.select(list(reference_columns)).to_pylist()

## Explore the held-out references

This block summarizes subject and diff lengths, date ranges, repository groups, and detected languages.

In [ ]:
def detect_languages(rows: list[dict[str, object]]) -> dict[str, int]:
    counts = Counter()
    for row in rows:
        for line in str(row["diff"]).splitlines():
            if line.startswith("+++ b/"):
                language = guess_language_all_methods(str(row["diff"]), file_name=line[6:])
                counts[language or "unknown"] += 1
    return dict(counts)


def split_summary(rows: list[dict[str, object]]) -> dict[str, object]:
    subjects = [str(row["subject"]) for row in rows]
    diffs = [str(row["diff"]) for row in rows]
    return {
        "rows": len(rows),
        "subject_length": {"min": min(map(len, subjects)), "max": max(map(len, subjects))},
        "diff_length": {"min": min(map(len, diffs)), "max": max(map(len, diffs))},
        "date_range": {
            "first": min(str(row["committed_at_utc"]) for row in rows),
            "last": max(str(row["committed_at_utc"]) for row in rows),
        },
        "repository_groups": len({row["repository_group_id"] for row in rows}),
        "languages": detect_languages(rows),
    }


eda_summary = {
    "validation": split_summary(validation_rows),
    "test": split_summary(test_rows),
}
eda_summary

## Build the audit queue

This block combines validation and test references into the queue that will be reviewed. Each entry keeps only the split, identifier, subject, and diff needed for the audit.

In [ ]:
audit_queue = [
    {"split": split_name, "id": row["id"], "subject": row["subject"], "diff": row["diff"]}
    for split_name, rows in (("validation", validation_rows), ("test", test_rows))
    for row in rows
]
len(audit_queue)